# Experiment 8: Bayesian Network - Heart Disease Diagnosis

## Overview
A Bayesian Network is a probabilistic graphical model that represents conditional dependencies between variables using a directed acyclic graph (DAG). In medical diagnosis, it can model relationships between symptoms, risk factors, and diseases.

## Key Concepts
- **Bayesian Network**: DAG with conditional probability tables
- **Nodes**: Random variables (symptoms, diseases, risk factors)
- **Edges**: Direct causal/dependency relationships
- **CPT**: Conditional Probability Tables
- **Inference**: Computing probabilities given evidence

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create synthetic heart disease data
np.random.seed(42)
n = 200
data = pd.DataFrame({
    'age': np.random.randint(30, 80, n),
    'cholesterol': np.random.randint(150, 400, n),
    'blood_pressure': np.random.randint(90, 180, n),
    'heart_rate': np.random.randint(60, 150, n),
    'disease': np.random.choice([0, 1], n, p=[0.7, 0.3])
})

# Add correlation
data.loc[data['age'] > 60, 'disease'] = np.where(
    np.random.random(len(data[data['age'] > 60])) > 0.5, 1, data.loc[data['age'] > 60, 'disease']
)

# Split
X = data.drop('disease', axis=1)
y = data['disease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Bayesian Network (using Naive Bayes as approximation)
bn = GaussianNB()
bn.fit(X_train, y_train)

# Predict
y_pred = bn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=['No Disease', 'Disease'])}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Disease', 'Disease'], yticklabels=['No Disease', 'Disease'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Heart Disease Prediction')
plt.show()

print(f"\nSensitivity (True Positive Rate): {cm[1,1]/(cm[1,1]+cm[1,0]):.4f}")
print(f"Specificity (True Negative Rate): {cm[0,0]/(cm[0,0]+cm[0,1]):.4f}")

## Load Heart Disease Dataset

In [ ]:
# Load heart disease dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'

try:
    # Try to load from URL
    df = pd.read_csv(url, header=None)
    print("Loaded heart disease dataset from UCI ML Repository")
except:
    # Fallback: create synthetic dataset
    print("Using synthetic heart disease dataset...")
    np.random.seed(42)
    n_samples = 300
    df = pd.DataFrame({
        'age': np.random.randint(29, 77, n_samples),
        'sex': np.random.randint(0, 2, n_samples),  # 0=F, 1=M
        'cp': np.random.randint(0, 4, n_samples),  # chest pain type
        'trestbps': np.random.randint(90, 200, n_samples),  # resting bp
        'chol': np.random.randint(150, 400, n_samples),  # cholesterol
        'fbs': np.random.randint(0, 2, n_samples),  # fasting blood sugar
        'restecg': np.random.randint(0, 3, n_samples),  # resting ecg
        'thalach': np.random.randint(60, 200, n_samples),  # max heart rate
        'exang': np.random.randint(0, 2, n_samples),  # exercise angina
        'oldpeak': np.random.uniform(0, 6, n_samples),  # ST depression
        'slope': np.random.randint(0, 3, n_samples),  # ST slope
        'ca': np.random.randint(0, 5, n_samples),  # major vessels
        'thal': np.random.randint(0, 4, n_samples),  # thalassemia
        'target': np.random.choice([0, 1], n_samples, p=[0.6, 0.4])  # 0=no disease, 1=disease
    })
    # Add some correlation
    df.loc[df['age'] > 60, 'target'] = np.where(
        np.random.random(len(df[df['age'] > 60])) > 0.5, 1, df.loc[df['age'] > 60, 'target']
    )

print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nDataset info:")
print(df.info())

## Data Preparation

In [ ]:
# Set column names if not already set
if len(df.columns) > 13:
    columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
               'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
    df.columns = columns[:len(df.columns)]

# Check for missing values
print("Missing values:")
print(df.isnull().sum())

# Remove rows with missing values if any
df = df.dropna()

# Convert target to binary (0 or 1)
df['target'] = (df['target'] > 0).astype(int)

print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"No disease: {(df['target']==0).sum()}")
print(f"Disease: {(df['target']==1).sum()}")

# Summary statistics
print(f"\nFeature statistics:")
print(df.describe())

## Simple Bayesian Network Model (Using Naive Bayes as approximation)

In [ ]:
# Prepare data
X = df.drop('target', axis=1)
y = df['target']

feature_names = X.columns.tolist()

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")
print(f"Features: {feature_names}")

# Train Naive Bayes model (approximation of Bayesian Network)
model = GaussianNB()
model.fit(X_train, y_train)

print("\nBayesian Network model trained!")

## Model Predictions and Evaluation

In [ ]:
# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)

# Accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("\n" + "="*60)
print("BAYESIAN NETWORK FOR HEART DISEASE DIAGNOSIS")
print("="*60)
print(f"\nTraining Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## Classification Report

In [ ]:
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, 
                          target_names=['No Disease', 'Disease']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix:")
print(cm)
print(f"\nTrue Negatives (correct no disease): {cm[0,0]}")
print(f"False Positives (incorrect disease): {cm[0,1]}")
print(f"False Negatives (missed disease): {cm[1,0]}")
print(f"True Positives (correct disease): {cm[1,1]}")

## Visualization

In [ ]:
# Plot confusion matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
           xticklabels=['No Disease', 'Disease'],
           yticklabels=['No Disease', 'Disease'])
ax1.set_ylabel('True Label')
ax1.set_xlabel('Predicted Label')
ax1.set_title('Confusion Matrix - Heart Disease Prediction')

# Accuracy comparison
metrics = ['Training', 'Test']
accuracies = [train_accuracy, test_accuracy]
colors = ['green', 'blue']
bars = ax2.bar(metrics, accuracies, color=colors, alpha=0.7, edgecolor='black')
ax2.set_ylabel('Accuracy')
ax2.set_ylim([0, 1])
ax2.set_title('Model Accuracy')
ax2.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Feature Analysis

In [ ]:
# Class means (important for understanding Bayesian Network)
class_means = []
for class_val in [0, 1]:
    means = X_train[y_train == class_val].mean()
    class_means.append(means)

comparison_df = pd.DataFrame({
    'No Disease': class_means[0],
    'Disease': class_means[1],
    'Difference': class_means[1] - class_means[0]
})

comparison_df = comparison_df.sort_values('Difference', key=abs, ascending=False)
print("\nFeature Comparison Between Classes:")
print(comparison_df.head(10))

# Plot top distinguishing features
fig, ax = plt.subplots(figsize=(10, 6))
top_features = comparison_df.head(10)
ax.barh(range(len(top_features)), top_features['Difference'], color='steelblue', alpha=0.7)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features.index)
ax.set_xlabel('Difference in Mean Values')
ax.set_title('Top 10 Distinguishing Features Between Disease Groups')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Example Diagnosis

In [ ]:
# Example patient data for diagnosis
example_patients = pd.DataFrame([
    {  # Patient 1: Healthy profile
        'age': 35, 'sex': 1, 'cp': 0, 'trestbps': 120, 'chol': 180,
        'fbs': 0, 'restecg': 0, 'thalach': 150, 'exang': 0,
        'oldpeak': 0, 'slope': 1, 'ca': 0, 'thal': 1
    },
    {  # Patient 2: At-risk profile
        'age': 65, 'sex': 1, 'cp': 3, 'trestbps': 160, 'chol': 300,
        'fbs': 1, 'restecg': 1, 'thalach': 110, 'exang': 1,
        'oldpeak': 3.5, 'slope': 0, 'ca': 2, 'thal': 3
    },
    {  # Patient 3: Moderate-risk profile
        'age': 50, 'sex': 0, 'cp': 1, 'trestbps': 140, 'chol': 220,
        'fbs': 0, 'restecg': 1, 'thalach': 130, 'exang': 0,
        'oldpeak': 1.5, 'slope': 1, 'ca': 1, 'thal': 2
    }
])

print("\n" + "="*60)
print("EXAMPLE PATIENT DIAGNOSES")
print("="*60)

predictions = model.predict(example_patients)
proba = model.predict_proba(example_patients)

for i, (pred, prob) in enumerate(zip(predictions, proba), 1):
    diagnosis = "Disease Present" if pred == 1 else "No Disease"
    confidence = prob[pred] * 100
    print(f"\nPatient {i}:")
    print(f"  Age: {int(example_patients.iloc[i-1]['age'])}")
    print(f"  Diagnosis: {diagnosis}")
    print(f"  Confidence: {confidence:.2f}%")
    print(f"  Risk Score: {prob[1]:.4f}")
    print(f"    No Disease: {prob[0]:.4f}")
    print(f"    Disease: {prob[1]:.4f}")

## Clinical Interpretation

In [ ]:
# Calculate sensitivity and specificity
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0  # Positive Predictive Value
npv = tn / (tn + fn) if (tn + fn) > 0 else 0  # Negative Predictive Value

print("\n" + "="*60)
print("CLINICAL PERFORMANCE METRICS")
print("="*60)
print(f"\nSensitivity (True Positive Rate): {sensitivity:.4f}")
print(f"  - Ability to correctly identify disease patients")
print(f"\nSpecificity (True Negative Rate): {specificity:.4f}")
print(f"  - Ability to correctly identify healthy individuals")
print(f"\nPositive Predictive Value (PPV): {ppv:.4f}")
print(f"  - Probability of disease given positive test")
print(f"\nNegative Predictive Value (NPV): {npv:.4f}")
print(f"  - Probability of no disease given negative test")

# Plot clinical metrics
fig, ax = plt.subplots(figsize=(10, 6))
metrics = ['Sensitivity\n(Recall)', 'Specificity', 'PPV\n(Precision)', 'NPV']
values = [sensitivity, specificity, ppv, npv]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = ax.bar(metrics, values, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Score')
ax.set_ylim([0, 1])
ax.set_title('Clinical Performance Metrics')
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{val:.4f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

## Viva Questions

### Basic Concepts
1. **What is a Bayesian Network?**
   - Probabilistic graphical model
   - Represents conditional dependencies between variables
   - Uses Directed Acyclic Graph (DAG) structure
   - Combines nodes (variables) and edges (dependencies)

2. **Explain the structure of a Bayesian Network for medical diagnosis.**
   - Nodes represent: symptoms, risk factors, diseases, findings
   - Edges show causal relationships or dependencies
   - Conditional Probability Tables (CPTs) store probabilities
   - Example: Risk factors → Symptoms → Disease

3. **What is Conditional Probability in medical context?**
   - P(Disease | Symptoms)
   - Probability of disease given observed symptoms
   - More informative than prior probability alone

### Implementation Questions
4. **How does Naive Bayes approximate a Bayesian Network?**
   - Assumes conditional independence of features given class
   - All features depend only on the target class
   - Simplifies computation (though assumption is naive)
   - Good approximation for many practical problems

5. **What is the role of CPT (Conditional Probability Table)?**
   - Stores P(child | parent) for each variable
   - Learned from training data
   - Used for inference and prediction
   - Can be based on expert knowledge or data

6. **How is inference performed in Bayesian Networks?**
   - Forward inference: predict disease given symptoms
   - Backward inference: predict likely symptoms given disease
   - Uses Bayes' theorem and conditional probabilities
   - Can handle partial evidence (some observations missing)

### Medical Applications
7. **What is sensitivity in medical diagnosis?**
   - True Positive Rate: TP / (TP + FN)
   - Ability to correctly identify disease patients
   - Important when missing disease is costly
   - Higher sensitivity means fewer false negatives

8. **What is specificity in medical diagnosis?**
   - True Negative Rate: TN / (TN + FP)
   - Ability to correctly identify healthy individuals
   - Important when false alarms are costly
   - Higher specificity means fewer false positives

9. **What is the trade-off between sensitivity and specificity?**
   - Improving one often decreases the other
   - Threshold adjustment changes the trade-off
   - Clinical requirements determine which to prioritize
   - ROC curve shows the trade-off

10. **How can Bayesian Networks improve medical decision-making?**
    - Provides probabilistic diagnosis (not just binary)
    - Handles uncertainty in medical data
    - Can incorporate expert knowledge and data
    - Explains reasoning through network structure
    - Supports what-if analysis (diagnostic reasoning)
    - Can recommend additional tests for uncertainty reduction